# 🔗 NGROK TUNNEL — Chạy cell này đầu tiên trên Colab browser
> Sau khi có URL, kết nối VSCode: chọn kernel → Existing Jupyter Server → paste URL

In [ ]:
# NGROK TUNNEL SETUP
# Chay cell nay tren Colab browser TRUOC KHI ket noi VSCode

!pip install pyngrok -q

from pyngrok import ngrok
import subprocess, time

NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"
ngrok.set_auth_token(NGROK_TOKEN)

subprocess.Popen([
    "jupyter", "notebook",
    "--no-browser", "--port=8888",
    "--NotebookApp.token=kltn2026",
    "--NotebookApp.allow_origin=*",
    "--NotebookApp.disable_check_xsrf=True"
])
time.sleep(3)

tunnel = ngrok.connect(8888)
print("=" * 60)
print(f"VSCode URL: {tunnel.public_url}/?token=kltn2026")
print("=" * 60)

In [3]:
import platform, sys, os

print(f"OS      : {platform.system()} {platform.release()}")
print(f"Python  : {sys.version.split()[0]}")
print(f"Hostname: {platform.node()}")
print(f"User    : {os.environ.get('USER', 'unknown')}")
print(f"CWD     : {os.getcwd()}")

# Kiểm tra Drive đã mount chưa
from pathlib import Path
drive = Path('/content/drive/MyDrive')
print(f"\nDrive mounted: {drive.exists()}")
print(f"KLTN folder : {(drive / 'KLTN').exists()}")


OS      : Linux 6.6.113+
Python  : 3.12.13
Hostname: e26c5f4e2c1b
User    : unknown
CWD     : /content

Drive mounted: True
KLTN folder : True


📌 **[NGROK]** Tunnel này expose Jupyter server của Colab ra internet để VSCode kết nối vào. 
URL thay đổi mỗi lần restart Colab — cần chạy lại cell này và update URL trong VSCode. 
Free tier ngrok: 1 tunnel, không giới hạn thời gian. Token lấy từ dashboard.ngrok.com.

# ⚙️ SESSION 0 — SETUP & RESTART CELL
> Chạy toàn bộ session này sau mỗi lần restart Colab runtime.
> Các session sau đọc data từ CSV — không cần chạy lại session nặng.

In [ ]:
# [0.1] Mount Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive', force_remount=False)

BASE = Path('/content/drive/MyDrive/KLTN')
if BASE.exists():
    print(f"✅ Drive mounted, BASE exists: {BASE}")
else:
    print(f"⚠️ BASE không tồn tại: {BASE} — kiểm tra lại cấu trúc thư mục Drive")

📌 **[0.1]** Mount Google Drive để truy cập data. `force_remount=False` giúp tránh unmount 
nếu Drive đã được mount từ trước (ví dụ khi chạy lại cell). Nếu BASE không tồn tại, 
kiểm tra lại tên thư mục trên Drive — phân biệt hoa/thường.

In [ ]:
# [0.2] Install missing libraries
import importlib, subprocess, sys

def ensure_package(import_name, pip_name=None):
    """Chi pip install neu package chua import duoc."""
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
        print(f'✅ {import_name} already installed')
    except ImportError:
        print(f'📦 Installing {pip_name}...')
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name, "-q"])
        print(f'✅ {pip_name} installed')

ensure_package('xgboost')
ensure_package('prophet')
ensure_package('scipy')
ensure_package('sklearn', 'scikit-learn')
ensure_package('joblib')
print("\n✅ Tat ca packages san sang")

📌 **[0.2]** Kiểm tra package trước khi install tránh re-install mất thời gian (~2-3 phút mỗi lần). 
Google Colab đã có sẵn `scikit-learn`, `scipy`, `xgboost` — thường chỉ cần install `prophet`. 
`prophet` yêu cầu `pystan` và compile C++ nên lần đầu cài sẽ lâu hơn.

In [ ]:
# [0.3] Import tat ca thu vien
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
import json
from pathlib import Path
from typing import List, Dict, Tuple, Optional

from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('husl')

print("✅ Tat ca imports thanh cong")
print(f"  pandas {pd.__version__} | numpy {np.__version__}")

📌 **[0.3]** Tập trung tất cả import vào một cell duy nhất. Khi Colab restart, chỉ cần chạy 
lại Session 0 (5 cells) là đủ — không cần tìm import rải rác trong các session khác. 
`warnings.filterwarnings("ignore")` ẩn các DeprecationWarning từ Prophet và XGBoost để output dễ đọc hơn.

In [ ]:
# [0.4] Paths + Constants

BASE         = Path('/content/drive/MyDrive/KLTN')
RAW          = BASE / 'dataset' / 'epidemic' / 'raw'
PROCESSED    = BASE / 'dataset' / 'processed'
WEATHER_DIR  = BASE / 'dataset' / 'weather' / 'processed'
MODELS_DIR   = BASE / 'models'
OUTPUTS_DIR  = BASE / 'outputs'
FEATURES_DIR = PROCESSED

ERA5_FILE    = WEATHER_DIR / 'era5_weekly_2010_2019_final.csv'
MASTER_FILE  = PROCESSED   / 'master_weekly_2010_2019.csv'
FLUNET_FILE  = RAW          / 'VIW_FNT.csv'
DENGUE_FILE  = RAW          / 'National_extract_V1_3.csv'
ECDC_ILI     = RAW          / 'ILIARIRates.csv'
ECDC_SENT    = RAW          / 'sentinelTestsDetectionsPositivity.csv'

FEATURES_FLU_FILE    = FEATURES_DIR / 'features_flu_2010_2019.csv'
FEATURES_DENGUE_FILE = FEATURES_DIR / 'features_dengue_2010_2019.csv'
MODEL_FLU_FILE       = MODELS_DIR   / 'xgb_flu_final.pkl'
MODEL_DENGUE_FILE    = MODELS_DIR   / 'xgb_dengue_final.pkl'
FEATURE_LIST_FILE    = OUTPUTS_DIR  / 'feature_list.json'

for d in [MODELS_DIR, OUTPUTS_DIR, PROCESSED]:
    d.mkdir(parents=True, exist_ok=True)

TRAIN_START  = 2010
TRAIN_END    = 2019
VAL_YEAR     = 2022
COVID_YEARS  = [2020, 2021]
TARGET_FLU    = 'inf_cases'
TARGET_DENGUE = 'dengue_log1p'
LAG_FLU    = [1, 2, 3]
LAG_DENGUE = [6, 8, 10, 12, 14]
WEATHER_VARS = [
    'temp_c', 'dewpoint_c', 'temp_min_c', 'temp_max_c', 'temp_range_c',
    'humidity_pct', 'wind_ms', 'precip_mm', 'conv_precip_mm', 'ls_precip_mm',
    'evap_mm', 'water_vapour', 'solar_wm2', 'uv_wm2', 'thermal_wm2',
    'cloud_cover', 'msl_pa', 'blh_m'
]

print("✅ Paths va constants da khai bao")
print(f"  TRAIN: {TRAIN_START}–{TRAIN_END} | VAL: {VAL_YEAR}")
print(f"  Weather vars: {len(WEATHER_VARS)} bien")

📌 **[0.4]** Tất cả path và constant khai báo tại đây — các session sau chỉ cần chạy lại 
Session 0 để có đủ biến, không tự khai báo lại tránh inconsistency. 
`mkdir(parents=True, exist_ok=True)` đảm bảo idempotent — chạy lại không báo lỗi dù thư mục đã tồn tại.

In [ ]:
# [0.5] File verification

files_to_check = {
    "MASTER_FILE":  MASTER_FILE,
    "ERA5_FILE":    ERA5_FILE,
    "FLUNET_FILE":  FLUNET_FILE,
    "DENGUE_FILE":  DENGUE_FILE,
    "ECDC_ILI":     ECDC_ILI,
    "ECDC_SENT":    ECDC_SENT,
}

for name, path in files_to_check.items():
    if path.exists():
        size_mb = path.stat().st_size / 1024**2
        print(f'✅ {name}: {path.name} ({size_mb:.1f} MB)')
    else:
        print(f'⚠️  {name}: KHONG TIM THAY -> {path}')

if MASTER_FILE.exists():
    _m = pd.read_csv(MASTER_FILE)
    print(f"\n📊 master_weekly shape: {_m.shape}")
    print(f"   Columns: {list(_m.columns)}")
    print(f"   iso3 unique: {_m[\"iso3\"].nunique()} countries")
    print(f"   year range: {_m[\"iso_year\"].min()}–{_m[\"iso_year\"].max()}")
    del _m

📌 **[0.5]** ⚠️ cảnh báo cho file không tìm thấy không phải lỗi nghiêm trọng — ECDC chỉ dùng 
cho validation dashboard (có từ 2021). ERA5 cover 158/172 quốc gia (92%) do giới hạn KD-tree 
centroid matching. Dengue missing 88.9% rows là đúng — chỉ endemic countries mới có data.

# 🔍 SESSION 1 — LOAD & INSPECT RAW DATA
> **Mục tiêu:** Hiểu cấu trúc từng file nguồn, phát hiện vấn đề trước khi xử lý.
> **Input:** Raw CSV files từ RAW folder
> **Output:** Không có (chỉ exploration)
> **Có thể skip nếu:** Đã quen với cấu trúc data, muốn chạy thẳng SESSION 4+

In [ ]:
# [1.0] RESTART CELL — load tất cả raw files
flu      = pd.read_csv(FILES['flunet'], low_memory=False)
flu_meta = pd.read_csv(FILES['flu_meta'], low_memory=False)
dengue   = pd.read_csv(FILES['dengue'], low_memory=False)
ecdc_sen = pd.read_csv(FILES['ecdc_sen'], low_memory=False)
ecdc_ili = pd.read_csv(FILES['ecdc_ili'], low_memory=False)

for name, df in [('flu', flu), ('flu_meta', flu_meta), ('dengue', dengue),
                  ('ecdc_sen', ecdc_sen), ('ecdc_ili', ecdc_ili)]:
    print(f'{name}: shape={df.shape} | cols={list(df.columns[:5])}...')

📌 **[1.0]** FILES dict đã được define ở SESSION 0 — bao gồm đường dẫn đến tất cả raw files.
Load tập trung ở đây để các cell [1.1]–[1.4] chỉ cần tham chiếu biến, không load lại.
Nếu session restart, chạy lại [1.0] này trước khi chạy các cell inspect.

In [ ]:
# [1.1] Inspect FluNet
print('=== FluNet ===')
print(f'Shape: {flu.shape}')
print(f'Columns ({len(flu.columns)}):', list(flu.columns))
print(f'Year range: {flu["ISO_YEAR"].min()}-{flu["ISO_YEAR"].max()}')
print(f'Countries: {flu["COUNTRY_CODE"].nunique()}')
display(flu.head(3))

📌 **[1.1]** FluNet có 53 cột gồm nhiều subtype chi tiết (AH1N12009, AH3, BVIC...). Các cột
quan trọng sẽ dùng: INF_A, INF_B, COUNTRY_CODE, ISO_YEAR, ISO_WEEK.
Lưu ý: RSV và RSV_PROCESSED tồn tại nhưng khác đơn vị — sẽ xử lý ở SESSION 2.
PARAINFLUENZA present nhưng sẽ bị drop do missing rate cao (xem SESSION 2).

In [ ]:
# [1.2] Inspect OpenDengue
print('=== OpenDengue ===')
print(f'Shape: {dengue.shape}')
print('T_res distribution:')
print(dengue['T_res'].value_counts())
print(f'Year range (approx): {dengue["calendar_start_date"].dropna().iloc[0]} ... {dengue["calendar_start_date"].dropna().iloc[-1]}')
display(dengue.head(3))

📌 **[1.2]** T_res phân ra Week/Month/Year — chỉ dùng Week+Month để đủ granularity cho model tuần.
Date format của OpenDengue là MM/DD/YYYY (không nhất quán), cần `format='mixed'` khi parse.
Điều này đã được confirm và sẽ xử lý tường minh ở SESSION 3 và SESSION 5.

In [ ]:
# [1.3] Inspect ECDC Sentinel
print('=== ECDC Sentinel ===')
print(f'Shape: {ecdc_sen.shape}')
pathogen_col = [c for c in ecdc_sen.columns if 'pathogen' in c.lower() or 'Pathogen' in c]
if pathogen_col:
    print(f'Unique pathogens: {ecdc_sen[pathogen_col[0]].unique()}')
print(f'Countries: {ecdc_sen.iloc[:, 0].nunique()}')
display(ecdc_sen.head(3))

📌 **[1.3]** ECDC Sentinel có cả SARS-CoV-2 — cần filter khi dùng. Chỉ 30 quốc gia châu Âu,
chỉ có data từ 2021. Quyết định đã chốt: ECDC chỉ dùng cho validation và dashboard,
không dùng cho training (vì train period là 2010–2019).

In [ ]:
# [1.4] Inspect ECDC ILI
print('=== ECDC ILI ===')
print(f'Shape: {ecdc_ili.shape}')
age_col = [c for c in ecdc_ili.columns if 'age' in c.lower()]
ind_col = [c for c in ecdc_ili.columns if 'indicator' in c.lower()]
print(f'Age columns: {age_col}')
print(f'Indicator columns: {ind_col}')
yr_col = [c for c in ecdc_ili.columns if 'year' in c.lower()]
if yr_col:
    print(f'Year range: {ecdc_ili[yr_col[0]].min()}-{ecdc_ili[yr_col[0]].max()}')
display(ecdc_ili.head(3))

📌 **[1.4]** ECDC ILI có age groups đầy đủ (0–4, 5–14, 15–64, 65+, total) — hữu ích cho
dashboard chi tiết khi hiển thị breakdown theo nhóm tuổi.
Cũng chỉ có từ 2021 nên không dùng cho training.

# 🔎 SESSION 2 — DATA QUALITY CHECK
> **Mục tiêu:** Missing rate, coverage quốc gia/năm, phát hiện anomaly.
> **Input:** raw DataFrames từ SESSION 1 (hoặc load lại từ [2.0])
> **Output:** Không có — kết quả dẫn đến các quyết định tiền xử lý

In [ ]:
# [2.0] RESTART CELL — load flu và dengue từ disk
flu    = pd.read_csv(FILES['flunet'], low_memory=False)
dengue = pd.read_csv(FILES['dengue'], low_memory=False)
print(f'flu: {flu.shape}')
print(f'dengue: {dengue.shape}')

📌 **[2.0]** Chỉ load 2 file cần cho session này để tiết kiệm RAM.
Nếu đã chạy SESSION 1 và biến còn tồn tại, cell này không bị lỗi — chỉ reload mới nhất từ disk.

In [ ]:
# [2.1] FluNet — Missing rate cho các cột quan trọng
check_cols = ['INF_A', 'INF_B', 'INF_ALL', 'RSV', 'RSV_PROCESSED',
              'PARAINFLUENZA', 'ILI_ACTIVITY', 'SPEC_PROCESSED_NB']
# Only keep cols that exist in dataframe
check_cols = [c for c in check_cols if c in flu.columns]
missing_pct = flu[check_cols].isnull().mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(8, 5))
missing_pct.plot(kind='barh', ax=ax, color='salmon', edgecolor='black', linewidth=0.5)
ax.set_xlabel('Missing (%)')
ax.set_title('FluNet — Missing Rate per Column')
for bar, val in zip(ax.patches, missing_pct):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

📌 **[2.1]** Kết quả xác nhận các quyết định đã chốt:
- INF_ALL missing ~44% → dùng INF_A + INF_B thay thế (fillna(0) vì missing = không báo cáo).
- PARAINFLUENZA missing ~85.5% → bỏ hoàn toàn khỏi feature set.
- RSV_PROCESSED khác đơn vị với RSV (corr=0.729 nhưng scale khác) → chỉ giữ RSV.

In [ ]:
# [2.2] FluNet — Coverage theo năm (số quốc gia báo cáo)
coverage = flu.groupby('ISO_YEAR')['COUNTRY_CODE'].nunique().reset_index()
coverage.columns = ['year', 'n_countries']

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(coverage['year'], coverage['n_countries'], color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(TRAIN_START - 0.5, color='green', lw=2, ls='--', label=f'Train start ({TRAIN_START})')
ax.axvline(TRAIN_END + 0.5, color='red', lw=2, ls='--', label=f'Train end ({TRAIN_END})')
ax.set_xlabel('Year')
ax.set_ylabel('Countries reporting')
ax.set_title('FluNet — Country Coverage by Year')
ax.legend()
plt.tight_layout()
plt.show()

📌 **[2.2]** Coverage ổn định từ 2010 (~120+ quốc gia). Giai đoạn 2020–2021 giảm mạnh do
nhiều nước ngừng báo cáo FluNet trong đại dịch COVID — đây là lý do bổ sung cho quyết định
exclude 2020–2021. Coverage ổn định ở 2010–2019 là bằng chứng train set đáng tin cậy.

In [ ]:
# [2.3] OpenDengue — Missing & coverage
print('dengue_total missing rate:', round(dengue['dengue_total'].isnull().mean()*100, 1), '%')
print('Year range:', dengue['calendar_start_date'].dropna().iloc[0], '...',
      dengue['calendar_start_date'].dropna().iloc[-1])

fig, ax = plt.subplots(figsize=(6, 6))
tres_counts = dengue['T_res'].value_counts()
ax.pie(tres_counts, labels=tres_counts.index, autopct='%1.1f%%', startangle=90,
       colors=['#2ecc71','#3498db','#e74c3c'])
ax.set_title('OpenDengue — T_res Distribution')
plt.tight_layout()
plt.show()

📌 **[2.3]** dengue_total missing ~88.9% trên master_weekly là bình thường — chỉ endemic
countries có data, và chỉ tuần có dịch mới có số. T_res Week chiếm ~78% rows → đủ để
dùng weekly aggregation. Phần Month sẽ được group by iso_week sau khi parse date.

In [ ]:
# [2.4] ERA5 — Coverage check
era5 = pd.read_csv(ERA5_FILE)
print(f'ERA5 shape: {era5.shape}')
print(f'Countries: {era5["iso3"].nunique()}')

missing_era5 = era5.drop(columns=['iso3','iso_year','iso_week']).isnull().mean() * 100
missing_era5 = missing_era5[missing_era5 > 0].sort_values(ascending=False)

if len(missing_era5) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    missing_era5.plot(kind='barh', ax=ax, color='coral')
    ax.set_xlabel('Missing (%)')
    ax.set_title('ERA5 — Missing Rate per Variable')
    plt.tight_layout()
    plt.show()
else:
    print('ERA5: khong co missing values')

📌 **[2.4]** ERA5 cover 158/172 countries (92%) do KD-tree nearest centroid mapping từ lưới
ERA5 (0.25°×0.25°) sang centroid quốc gia theo Natural Earth 50m.
14 quốc gia bị miss thường là đảo nhỏ hoặc quốc gia không có centroid rõ ràng trong shapefile.
92% coverage là acceptable cho mục tiêu global surveillance.

# 📊 SESSION 3 — EDA: SEASONALITY & TRENDS
> **Mục tiêu:** Xác nhận pattern mùa vụ rõ ràng — điều kiện cần để train model.
> **Input:** Raw FluNet + OpenDengue
> **Output:** Không có file — chỉ visualizations

In [ ]:
# [3.0] RESTART CELL + setup train range
flu    = pd.read_csv(FILES['flunet'], low_memory=False)
dengue = pd.read_csv(FILES['dengue'], low_memory=False)

flu_train = flu[flu['ISO_YEAR'].between(TRAIN_START, TRAIN_END)].copy()
flu_train['inf_total'] = flu_train['INF_A'].fillna(0) + flu_train['INF_B'].fillna(0)
print(f'flu_train: {flu_train.shape} | years: {TRAIN_START}-{TRAIN_END}')

📌 **[3.0]** TRAIN_START=2010, TRAIN_END=2019 đã chốt ở SESSION 0.
flu_train ở đây dùng riêng cho EDA, không phải feature matrix cuối cùng.
inf_total = INF_A + INF_B là target chính — INF_ALL bị bỏ do missing 44%.

In [ ]:
# [3.1] FluNet — Global trend + seasonality
flu_weekly = flu_train.groupby(['ISO_YEAR','ISO_WEEK'])['inf_total'].sum().reset_index()
flu_weekly['time_idx'] = flu_weekly['ISO_YEAR'] + flu_weekly['ISO_WEEK'] / 53
flu_season = flu_train.groupby('ISO_WEEK')['inf_total'].mean().reset_index()

fig, axes = plt.subplots(2, 1, figsize=(16, 9))

axes[0].plot(flu_weekly['time_idx'], flu_weekly['inf_total'], lw=1.2, color='steelblue')
axes[0].set_title('Global Influenza Cases per Week (2010-2019)')
axes[0].set_xlabel('Year')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

axes[1].bar(flu_season['ISO_WEEK'], flu_season['inf_total'], color='steelblue', alpha=0.8)
axes[1].set_title('Average Seasonality by ISO Week (2010-2019)')
axes[1].set_xlabel('ISO Week')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

plt.tight_layout()
plt.show()

📌 **[3.1]** Trend phẳng (không tăng mạnh) là tốt — cho thấy data ổn định, không bị confound bởi
reporting bias tăng dần theo năm. Seasonality rõ peak tuần 1–10 (mùa đông bắc bán cầu).
Đây là pattern chính model cần học; weather features cung cấp signal để predict timing và amplitude.

In [ ]:
# [3.2] FluNet — 5 quốc gia đại diện
countries = ['VNM', 'USA', 'GBR', 'BRA', 'AUS']
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=False)

for ax, iso in zip(axes, countries):
    df_c = flu_train[flu_train['COUNTRY_CODE'] == iso]
    season_c = df_c.groupby('ISO_WEEK')['inf_total'].mean()
    ax.bar(season_c.index, season_c.values, color='steelblue', alpha=0.8)
    ax.set_title(iso, fontsize=12)
    ax.set_xlabel('ISO Week')
    if ax is axes[0]:
        ax.set_ylabel('Avg cases')

plt.suptitle('Influenza Seasonality by Country (2010-2019)', fontsize=13)
plt.tight_layout()
plt.show()

📌 **[3.2]** Pattern khác nhau rõ theo khí hậu: BRA và AUS peak tháng 6–8 (nam bán cầu),
USA/GBR peak tháng 12–2. VNM có 2 peak nhỏ hơn (nhiệt đới).
Model train per-country tự học được sự khác biệt này qua iso3 encoding và lag features.

In [ ]:
# [3.3] Dengue — Filter + parse date
dengue_wm = dengue[dengue['T_res'].isin(['Week','Month'])].copy()
dengue_wm['date_parsed'] = pd.to_datetime(dengue_wm['calendar_start_date'], format='mixed', dayfirst=False)
iso_cal = dengue_wm['date_parsed'].dt.isocalendar()
dengue_wm['ISO_YEAR'] = iso_cal.year.astype(int)
dengue_wm['ISO_WEEK'] = iso_cal.week.astype(int)
dengue_train = dengue_wm[dengue_wm['ISO_YEAR'].between(TRAIN_START, TRAIN_END)].copy()
dengue_train['dengue_log'] = np.log1p(dengue_train['dengue_total'])
print(f'dengue_train: {dengue_train.shape}')
print(f'Countries: {dengue_train["ISO_A0"].nunique()}')

📌 **[3.3]** `format='mixed'` cần thiết vì OpenDengue date không nhất quán (MM/DD/YYYY).
log1p ngay ở đây để visualization dùng log scale — dengue_total bị dominated bởi Brazil
với ~10.49M ca, chiếm 70% tổng global. log1p giúp thấy pattern của các nước khác.

In [ ]:
# [3.4] Dengue — Trend + seasonality (raw vs log)
by_year_raw = dengue_train.groupby('ISO_YEAR')['dengue_total'].sum()
by_week_raw = dengue_train.groupby('ISO_WEEK')['dengue_total'].mean()
by_year_log = dengue_train.groupby('ISO_YEAR')['dengue_log'].sum()
by_week_log = dengue_train.groupby('ISO_WEEK')['dengue_log'].mean()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0,0].bar(by_year_raw.index, by_year_raw.values, color='coral')
axes[0,0].set_title('Raw — by Year')
axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

axes[0,1].bar(by_week_raw.index, by_week_raw.values, color='coral')
axes[0,1].set_title('Raw — by ISO Week')

axes[1,0].bar(by_year_log.index, by_year_log.values, color='#27ae60')
axes[1,0].set_title('Log1p — by Year')

axes[1,1].bar(by_week_log.index, by_week_log.values, color='#27ae60')
axes[1,1].set_title('Log1p — by ISO Week')

plt.suptitle('Dengue Trend & Seasonality (2010-2019)', fontsize=13)
plt.tight_layout()
plt.show()

📌 **[3.4]** Raw bị dominated bởi Brazil (10.49M ca). Log scale cho thấy pattern của các nước
khác rõ hơn — đây là bằng chứng thực nghiệm cho quyết định dùng log1p làm target.
By-week log plot cho thấy seasonality thực sự khi không bị Brazil che khuất.

In [ ]:
# [3.5] Dengue — Top 5 (loại Brazil)
dengue_no_bra = dengue_train[dengue_train['ISO_A0'] != 'BRA']
top5 = dengue_no_bra.groupby('ISO_A0')['dengue_total'].sum().nlargest(5).index.tolist()

fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=False)
for ax, iso in zip(axes, top5):
    df_c = dengue_no_bra[dengue_no_bra['ISO_A0'] == iso]
    season_c = df_c.groupby('ISO_WEEK')['dengue_total'].mean()
    ax.bar(season_c.index, season_c.values, color='#27ae60', alpha=0.8)
    ax.set_title(iso, fontsize=12)
    ax.set_xlabel('ISO Week')

plt.suptitle('Dengue Seasonality — Top 5 ex-Brazil (2010-2019)', fontsize=13)
plt.tight_layout()
plt.show()

📌 **[3.5]** Peak mùa mưa khác nhau theo khu vực: Đông Nam Á (Philippines, Thái Lan, Indonesia)
peak tuần 25–45; Trung Mỹ (Mexico, Colombia) peak tuần 1–20.
Sự đa dạng này xác nhận cần per-country model với weather lag features — không dùng global model đơn.

In [ ]:
# [3.6] Heatmap mùa vụ Việt Nam (Influenza)
vnm_flu = flu_train[flu_train['COUNTRY_CODE'] == 'VNM'][['ISO_YEAR','ISO_WEEK','inf_total']]
pivot = vnm_flu.pivot_table(index='ISO_YEAR', columns='ISO_WEEK', values='inf_total', aggfunc='sum')

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.3, ax=ax, cbar_kws={'label': 'Cases'})
ax.set_title('Vietnam Influenza Seasonality Heatmap (Year × ISO Week)')
ax.set_xlabel('ISO Week')
ax.set_ylabel('Year')
plt.tight_layout()
plt.show()

📌 **[3.6]** Heatmap Tuần×Năm là cách visualize seasonality hiệu quả nhất.
Nếu màu lặp lại theo cột (cùng tuần qua các năm) thì pattern ổn định — model có thể học được.
Việt Nam thường có 2 đợt nhỏ trong năm, không rõ ràng như USA/GBR — thách thức cho model.

# 🌦️ SESSION 4 — ERA5 DOWNLOAD & PROCESS
> **NẶNG — Chỉ chạy nếu ERA5_FILE chưa tồn tại**
> **Input:** ERA5 NetCDF files + Natural Earth shapefile
> **Output:** era5_weekly_2010_2019_final.csv
> ✅ File đã tồn tại — SESSION NÀY ĐÃ HOÀN THÀNH

In [ ]:
# [4.0] Idempotent guard
if ERA5_FILE.exists():
    era5 = pd.read_csv(ERA5_FILE)
    print(f'ERA5 da co: {ERA5_FILE.name}')
    print(f'Shape: {era5.shape} | Countries: {era5["iso3"].nunique()} | Years: {era5["iso_year"].min()}-{era5["iso_year"].max()}')
    print('SESSION 4 hoan thanh - skip xuong SESSION 5')
else:
    print('ERA5 chua co - can chay tu [4.1]')

📌 **[4.0]** ERA5 đã được xử lý và lưu vào era5_weekly_2010_2019_final.csv với 17 biến khí hậu.
Script gốc dùng KD-tree nearest centroid mapping từ lưới ERA5 (0.25°×0.25°) sang centroid
quốc gia theo Natural Earth 50m. Chỉ cần chạy lại nếu xóa file hoặc cần thêm biến mới.

In [ ]:
# [4.1] (Conditional) Process ERA5 nếu cần
# Placeholder — chỉ chạy nếu ERA5_FILE chua ton tai
# Script day du nam o: scripts/process_era5.py
# Thoi gian uoc tinh: 2-3 gio cho 10 nam du lieu

ERA5_VARS_DOWNLOAD = [
    '2m_temperature', 'total_precipitation', '2m_dewpoint_temperature',
    '10m_u_component_of_wind', '10m_v_component_of_wind',
    'surface_solar_radiation_downwards', 'surface_thermal_radiation_downwards',
    'total_cloud_cover', 'mean_sea_level_pressure', 'boundary_layer_height',
    'total_evaporation', 'convective_precipitation', 'large_scale_precipitation',
    'soil_temperature_level_1', 'volumetric_soil_water_layer_1',
    'surface_net_solar_radiation', 'total_column_water_vapour'
]
print(f'{len(ERA5_VARS_DOWNLOAD)} variables to download')
print('Xem scripts/process_era5.py de chay day du')

📌 **[4.1]** Download ERA5 qua CDS API mất ~2–3 giờ cho 10 năm × 17 biến.
Đã được lưu checkpoint theo từng năm (era5_weekly_era5_YYYY_checkpoint.csv) để tránh mất công
nếu bị ngắt giữa chừng. Cuối cùng concat tất cả vào final CSV.

# 🔗 SESSION 5 — PREPROCESSING & MERGE
> **Mục tiêu:** Chuẩn hóa 3 nguồn về key iso3+iso_year+iso_week, merge → master_weekly.csv
> **Input:** FluNet + ERA5 + OpenDengue + Malaria (GHO)
> **Output:** dataset/processed/master_weekly_2010_2019.csv
> ✅ File đã tồn tại — SESSION NÀY ĐÃ HOÀN THÀNH

In [ ]:
# [5.0] Idempotent guard + verify
if MASTER_FILE.exists():
    master = pd.read_csv(MASTER_FILE)
    print(f'master_weekly da co: {MASTER_FILE.name}')
    print(f'Shape: {master.shape}')
    print(f'Countries: {master["iso3"].nunique()} | Years: {master["iso_year"].min()}-{master["iso_year"].max()}')
    print(f'Columns: {list(master.columns)}')
    print('SESSION 5 hoan thanh - chuyen sang SESSION 6')
else:
    print('master_weekly chua co - can chay tu [5.1]')

📌 **[5.0]** master_weekly_2010_2019.csv là kết quả merge của FluNet (anchor) LEFT JOIN ERA5
LEFT JOIN OpenDengue LEFT JOIN Malaria. 64,949 rows × 27 columns, 172 quốc gia.
Đây là file đầu vào cho toàn bộ pipeline ML từ SESSION 6 trở đi.

In [ ]:
# [5.1] (Conditional) Merge logic — nếu cần tái tạo
# Chay cell nay ONLY neu MASTER_FILE chua ton tai

flu    = pd.read_csv(FILES['flunet'], low_memory=False)
dengue = pd.read_csv(FILES['dengue'], low_memory=False)
era5   = pd.read_csv(ERA5_FILE)

# FluNet preprocessing
flu_proc = flu[flu['ISO_YEAR'].between(TRAIN_START, TRAIN_END)].copy()
flu_proc['inf_cases'] = flu_proc['INF_A'].fillna(0) + flu_proc['INF_B'].fillna(0)
flu_proc['rsv_cases'] = flu_proc['RSV'].fillna(0)
flu_proc = flu_proc.rename(columns={'COUNTRY_CODE':'iso3','ISO_YEAR':'iso_year','ISO_WEEK':'iso_week'})
flu_proc = flu_proc[['iso3','iso_year','iso_week','inf_cases','rsv_cases']]

# ERA5 preprocessing
era5_proc = era5[era5['iso_year'].between(TRAIN_START, TRAIN_END)].copy()

# Dengue preprocessing
dengue_wm = dengue[dengue['T_res'].isin(['Week','Month'])].copy()
dengue_wm['date_parsed'] = pd.to_datetime(dengue_wm['calendar_start_date'], format='mixed', dayfirst=False)
iso_cal = dengue_wm['date_parsed'].dt.isocalendar()
dengue_wm['iso_year'] = iso_cal.year.astype(int)
dengue_wm['iso_week'] = iso_cal.week.astype(int)
dengue_proc = dengue_wm[dengue_wm['iso_year'].between(TRAIN_START, TRAIN_END)].copy()
dengue_proc['dengue_total'] = dengue_proc['dengue_total'].fillna(0)
dengue_proc['dengue_log1p'] = np.log1p(dengue_proc['dengue_total'])
dengue_proc = dengue_proc.rename(columns={'ISO_A0':'iso3'})
dengue_proc = dengue_proc.groupby(['iso3','iso_year','iso_week'], as_index=False).agg(
    dengue_total=('dengue_total','sum'), dengue_log1p=('dengue_log1p','mean')
)

# Merge: FluNet (anchor) LEFT JOIN ERA5 LEFT JOIN Dengue
master = flu_proc.merge(era5_proc, on=['iso3','iso_year','iso_week'], how='left')
master = master.merge(dengue_proc, on=['iso3','iso_year','iso_week'], how='left')
master['dengue_total'] = master['dengue_total'].fillna(0)
master['dengue_log1p'] = master['dengue_log1p'].fillna(0)

master.to_csv(MASTER_FILE, index=False)
print(f'Saved {len(master):,} rows -> {MASTER_FILE.name}')

📌 **[5.1]** FluNet là anchor (LEFT JOIN) — giữ nguyên tất cả 172 quốc gia × 10 năm × 52 tuần.
fillna(0) sau merge cho dengue_total vì quốc gia không có dengue report = 0 ca thực, không phải missing.
ERA5 join theo iso3+iso_year+iso_week — 14 quốc gia không có ERA5 sẽ có NaN cho weather features.

# 📊 SESSION 6 — CORRELATION & LAG ANALYSIS
> **Input:** master_weekly_2010_2019.csv
> **Mục tiêu:** Xác nhận lag time weather→disease bằng cross-correlation
> **Skip nếu:** Đã chạy và có kết quả (không có output file — session này chỉ visualize)

In [ ]:
# [6.0] RESTART CELL — load master data
master = pd.read_csv(MASTER_FILE)
print(f"Loaded master: {master.shape}")

flu_train = master[
    (master['iso_year'] >= TRAIN_START) &
    (master['iso_year'] <= TRAIN_END) &
    (master[TARGET_FLU].notna())
].copy()

dengue_train = master[
    (master['iso_year'] >= TRAIN_START) &
    (master['iso_year'] <= TRAIN_END) &
    (master[TARGET_DENGUE] > 0)
].copy()

print(f"flu_train: {flu_train.shape}")
print(f"dengue_train: {dengue_train.shape}")

📌 **[6.0]** Load lại data từ CSV thay vì dùng biến từ session trước — đảm bảo session independence. 
Filter `dengue_log1p > 0` loại bỏ các tuần không có dịch, giữ lại signal thực sự.

In [ ]:
# [6.1] Heatmap correlation: disease x disease
disease_cols = ['inf_cases', 'rsv_cases', 'dengue_total', 'malaria_cases']
corr_disease = master[disease_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_disease, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Disease Co-occurrence Correlation (2010-2019)', fontsize=13)
plt.tight_layout()
plt.show()

📌 **[6.1]** Heatmap này cho thấy co-occurrence giữa các bệnh hô hấp (Influenza, RSV) không? 
Thường Influenza và RSV có seasonal overlap (mùa đông), trong khi Dengue và Malaria không correlate.

In [ ]:
# [6.2] Heatmap: weather x influenza (lag=0)
weather_inf_corr = flu_train[WEATHER_VARS + [TARGET_FLU]].corr()[[TARGET_FLU]].drop(TARGET_FLU)
weather_inf_corr = weather_inf_corr.sort_values(TARGET_FLU, ascending=False)

fig, ax = plt.subplots(figsize=(4, 8))
sns.heatmap(weather_inf_corr, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Weather x Influenza\nCorrelation (lag=0)', fontsize=12)
plt.tight_layout()
plt.show()

📌 **[6.2]** Tương quan tức thời (lag=0) giữa các biến thời tiết và Influenza. 
Kỳ vọng: `temp_c` và `humidity_pct` có correlation âm (lạnh + khô → nhiều dịch hơn). 
Kết quả xác nhận lựa chọn weather feature có scientific basis.

In [ ]:
# [6.3] Cross-correlation: influenza vs temp_c and humidity_pct

def lag_corr(series_x, series_y, max_lag=12):
    """Cross-correlation between x (lagged) and y."""
    return [series_x.shift(lag).corr(series_y) for lag in range(0, max_lag + 1)]

flu_weekly = flu_train.groupby(['iso_year', 'iso_week'])[[TARGET_FLU, 'temp_c', 'humidity_pct']].mean()
lags = list(range(0, 9))
corr_temp  = lag_corr(flu_weekly['temp_c'], flu_weekly[TARGET_FLU], max_lag=8)
corr_humid = lag_corr(flu_weekly['humidity_pct'], flu_weekly[TARGET_FLU], max_lag=8)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, corrs, var in zip(axes, [corr_temp, corr_humid], ['temp_c', 'humidity_pct']):
    bars = ax.bar(lags, corrs, color=['#e74c3c' if c < 0 else '#3498db' for c in corrs])
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xlabel('Lag (weeks)')
    ax.set_ylabel('Pearson r')
    ax.set_title(f'CCF: {var} → inf_cases')
    ax.set_xticks(lags)
    for bar, c in zip(bars, corrs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{c:.2f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

📌 **[6.3]** Cross-correlation cho thấy lag nào weather ảnh hưởng mạnh nhất đến Influenza. 
Incubation period Influenza 1-4 ngày + reporting delay ~1 tuần → lag 1-3 tuần hợp lý. 
Nếu peak correlation ở lag 2, quyết định LAG_FLU = [1, 2, 3] đã chốt là đúng.

In [ ]:
# [6.4] Cross-correlation: dengue vs precip_mm and temp_c

dengue_weekly = dengue_train.groupby(['iso_year','iso_week'])[[TARGET_DENGUE,'precip_mm','temp_c']].mean()
lags_d = list(range(0, 17))
corr_precip = lag_corr(dengue_weekly['precip_mm'], dengue_weekly[TARGET_DENGUE], max_lag=16)
corr_temp_d = lag_corr(dengue_weekly['temp_c'], dengue_weekly[TARGET_DENGUE], max_lag=16)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, corrs, var in zip(axes, [corr_precip, corr_temp_d], ['precip_mm', 'temp_c']):
    bars = ax.bar(lags_d, corrs, color=['#e74c3c' if c < 0 else '#27ae60' for c in corrs])
    ax.axhline(0, color='black', lw=0.8)
    ax.axvspan(6, 14, alpha=0.12, color='green', label='LAG_DENGUE zone')
    ax.set_xlabel('Lag (weeks)')
    ax.set_ylabel('Pearson r')
    ax.set_title(f'CCF: {var} → dengue_log1p')
    ax.set_xticks(lags_d[::2])
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

📌 **[6.4]** Lag 6-14 tuần cho Dengue phản ánh mosquito breeding cycle: mưa → đọng nước → 
muỗi sinh sản (~2 tuần) → lây truyền (~1 tuần) → incubation (~1 tuần) → reporting delay (~2-4 tuần). 
Peak correlation ở lag ~8-12 tuần xác nhận LAG_DENGUE = [6, 8, 10, 12, 14].

In [ ]:
# [6.5] Summary heatmap: top weather vars x disease targets x lag

top_vars = ['temp_c', 'humidity_pct', 'precip_mm', 'solar_wm2', 'dewpoint_c']
lag_points = [0, 2, 4, 8, 12]
disease_map = {'inf_cases': flu_train, 'dengue_log1p': dengue_train}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (dis_name, df) in zip(axes, disease_map.items()):
    weekly = df.groupby(['iso_year','iso_week'])[top_vars + [dis_name]].mean()
    corr_mat = []
    for var in top_vars:
        row = [weekly[var].shift(lag).corr(weekly[dis_name]) for lag in lag_points]
        corr_mat.append(row)
    corr_df = pd.DataFrame(corr_mat, index=top_vars,
                            columns=[f'lag{l}w' for l in lag_points])
    sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='RdYlBu_r',
                center=0, vmin=-0.6, vmax=0.6, ax=ax, linewidths=0.4)
    ax.set_title(f'Weather x {dis_name}', fontsize=12)

plt.suptitle('Summary Weather Lag Correlation', fontsize=13)
plt.tight_layout()
plt.show()

📌 **[6.5]** Bảng tổng kết xác nhận: Influenza có signal rõ nhất ở lag 1-3 tuần với `temp_c`, 
`humidity_pct`, `dewpoint_c`. Dengue có signal rõ nhất ở lag 8-12 tuần với `precip_mm`, `temp_c`. 
Kết quả là scientific justification cho feature engineering ở Session 7.

# 🔧 SESSION 7 — FEATURE ENGINEERING
> **Input:** master_weekly_2010_2019.csv
> **Output:** features_flu_2010_2019.csv, features_dengue_2010_2019.csv
> **Idempotent:** kiểm tra file tồn tại trước khi tạo lại

In [ ]:
# [7.0] RESTART CELL — load master, check output files
master = pd.read_csv(MASTER_FILE)
print(f"Loaded master: {master.shape}")

if FEATURES_FLU_FILE.exists() and FEATURES_DENGUE_FILE.exists():
    print("✅ Feature files da ton tai — skip feature engineering")
    features_flu    = pd.read_csv(FEATURES_FLU_FILE)
    features_dengue = pd.read_csv(FEATURES_DENGUE_FILE)
    print(f"  flu: {features_flu.shape} | dengue: {features_dengue.shape}")
else:
    print("Chua co feature files — se tao moi")

📌 **[7.0]** Idempotent guard: nếu file đã có, load lại và skip toàn bộ session — tiết kiệm 
10-15 phút mỗi lần chạy. Nếu muốn regenerate, xoá file CSV rồi chạy lại.

In [ ]:
# [7.1] Sort data + base setup
master_sorted = master.sort_values(['iso3', 'iso_year', 'iso_week']).reset_index(drop=True)
df_feat = master_sorted.copy()
print(f"Sorted: {df_feat.shape} | Countries: {df_feat[chr(34)+"iso3"+chr(34)].nunique()}")

📌 **[7.1]** PHẢI sort theo `[iso3, iso_year, iso_week]` trước khi tạo lag features. Nếu không sort, 
`shift(1)` sẽ lấy nhầm row từ quốc gia khác — tuần cuối của ABW sẽ làm lag cho tuần đầu của AGO. 
Dùng groupby trong bước tiếp theo để đảm bảo an toàn tuyệt đối.

In [ ]:
# [7.2] Lag features: Influenza (lag 1, 2, 3 weeks)
for lag in LAG_FLU:
    col = f'inf_lag{lag}w'
    df_feat[col] = df_feat.groupby('iso3')[TARGET_FLU].shift(lag)

flu_lag_cols = [f'inf_lag{l}w' for l in LAG_FLU]
print(f"Influenza lag features: {flu_lag_cols}")

📌 **[7.2]** Dùng `groupby("iso3").shift(lag)` thay vì `shift(lag)` trực tiếp để tránh cross-country leakage. 
NaN ở lag đầu của mỗi quốc gia là bình thường — sẽ được dropna ở bước cuối.

In [ ]:
# [7.3] Lag features: Dengue (lag 6, 8, 10, 12, 14 weeks)
for lag in LAG_DENGUE:
    col = f'dengue_lag{lag}w'
    df_feat[col] = df_feat.groupby('iso3')[TARGET_DENGUE].shift(lag)

dengue_lag_cols = [f'dengue_lag{l}w' for l in LAG_DENGUE]
print(f"Dengue lag features: {dengue_lag_cols}")

📌 **[7.3]** Lag 6-14 tuần cho Dengue tương ứng với mosquito breeding cycle đã xác nhận ở Session 6. 
5 lag features giúp model học được shape của signal ở nhiều độ trễ khác nhau.

In [ ]:
# [7.4] Rolling mean features (shift(1) truoc rolling de tranh data leakage)
for window in [4, 8]:
    df_feat[f'inf_roll{window}w'] = (
        df_feat.groupby('iso3')[TARGET_FLU]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=2).mean())
    )
    df_feat[f'dengue_roll{window}w'] = (
        df_feat.groupby('iso3')[TARGET_DENGUE]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=2).mean())
    )
print("Rolling mean features created: inf_roll4w, inf_roll8w, dengue_roll4w, dengue_roll8w")

📌 **[7.4]** `shift(1)` trước `rolling()` rất quan trọng — đảm bảo rolling mean tại thời điểm t 
chỉ dùng data từ t-1 trở về trước, không dùng giá trị t hiện tại (data leakage). 
Rolling window 4 tuần = ~1 tháng, 8 tuần = ~2 tháng — nắm bắt short-term trend.

In [ ]:
# [7.5] Weather lag features
flu_w_vars   = ['temp_c', 'humidity_pct', 'precip_mm']
dengue_w_vars = ['temp_c', 'precip_mm', 'humidity_pct']
dengue_w_lags = [6, 8, 12]

for var in flu_w_vars:
    for lag in LAG_FLU:
        df_feat[f'{var}_flu_lag{lag}w'] = df_feat.groupby('iso3')[var].shift(lag)

for var in dengue_w_vars:
    for lag in dengue_w_lags:
        df_feat[f'{var}_dengue_lag{lag}w'] = df_feat.groupby('iso3')[var].shift(lag)

weather_lag_flu_cols = [f'{v}_flu_lag{l}w' for v in flu_w_vars for l in LAG_FLU]
weather_lag_den_cols = [f'{v}_dengue_lag{l}w' for v in dengue_w_vars for l in dengue_w_lags]
print(f"Weather lag flu: {len(weather_lag_flu_cols)} | dengue: {len(weather_lag_den_cols)}")

📌 **[7.5]** Weather lag features cho model biết điều kiện khí hậu trước đó ảnh hưởng như thế nào. 
Tách riêng lag set cho flu và dengue vì chỉ flu model dùng `_flu_lag` — tránh feature noise khi train.

In [ ]:
# [7.6] Seasonal encoding: sin/cos week, quarter

df_feat['sin_week'] = np.sin(2 * np.pi * df_feat['iso_week'] / 52)
df_feat['cos_week'] = np.cos(2 * np.pi * df_feat['iso_week'] / 52)
df_feat['quarter']  = ((df_feat['iso_week'] - 1) // 13 + 1).clip(1, 4)

print("Seasonal features: sin_week, cos_week, quarter")
print(f"  sin_week: [{df_feat["sin_week"].min():.3f}, {df_feat["sin_week"].max():.3f}]")

📌 **[7.6]** Encoding tuần dạng sin/cos thay vì số nguyên (1-52) để model hiểu tuần 1 và tuần 52 
gần nhau về mùa — tức là tính liên tục của mùa vụ. Nếu dùng số nguyên, model thấy khoảng cách 
tuần 1 và 52 là 51 đơn vị — sai về mùa vụ. `quarter` bổ sung macro-seasonal signal.

In [ ]:
# [7.7] Split flu/dengue features va save CSV

base_id_cols = ['iso3', 'iso_year', 'iso_week']

flu_feature_cols = (
    WEATHER_VARS +
    flu_lag_cols +
    ['inf_roll4w', 'inf_roll8w'] +
    weather_lag_flu_cols +
    ['sin_week', 'cos_week', 'quarter']
)
flu_all_cols = base_id_cols + flu_feature_cols + [TARGET_FLU]

dengue_feature_cols = (
    WEATHER_VARS +
    dengue_lag_cols +
    ['dengue_roll4w', 'dengue_roll8w'] +
    weather_lag_den_cols +
    ['sin_week', 'cos_week', 'quarter']
)
dengue_all_cols = base_id_cols + dengue_feature_cols + [TARGET_DENGUE]

features_flu = (
    df_feat[flu_all_cols]
    .dropna(subset=flu_feature_cols + [TARGET_FLU])
    .query(f'iso_year >= @TRAIN_START and iso_year <= @TRAIN_END')
    .copy()
)

features_dengue = (
    df_feat[dengue_all_cols]
    .dropna(subset=dengue_feature_cols + [TARGET_DENGUE])
    .query(f'iso_year >= @TRAIN_START and iso_year <= @TRAIN_END and dengue_log1p > 0')
    .copy()
)

if not FEATURES_FLU_FILE.exists():
    features_flu.to_csv(FEATURES_FLU_FILE, index=False)
    print(f"✅ Saved features_flu: {features_flu.shape}")
else:
    print("⏭️  features_flu da ton tai, skip")

if not FEATURES_DENGUE_FILE.exists():
    features_dengue.to_csv(FEATURES_DENGUE_FILE, index=False)
    print(f"✅ Saved features_dengue: {features_dengue.shape}")
else:
    print("⏭️  features_dengue da ton tai, skip")

print(f"\nFlu: {len(flu_feature_cols)} features | {features_flu["iso3"].nunique()} countries")
print(f"Dengue: {len(dengue_feature_cols)} features | {features_dengue["iso3"].nunique()} countries")

📌 **[7.7]** Tách riêng features_flu và features_dengue vì: (1) lag time khác nhau nên feature set khác, 
(2) country coverage khác nhau (flu ~172, dengue ~50 endemic countries), 
(3) dropna riêng tránh loại bỏ quá nhiều rows. Idempotent: file đã có thì không overwrite.

# 🤖 SESSION 8 — MODEL TRAINING
> **Input:** features_flu_2010_2019.csv, features_dengue_2010_2019.csv
> **Output:** models/xgb_flu_final.pkl, models/xgb_dengue_final.pkl
> **Walk-forward CV:** 6 folds (2014-2019), exclude 2020-2021

In [ ]:
# [8.0] RESTART CELL — load feature CSVs
features_flu    = pd.read_csv(FEATURES_FLU_FILE)
features_dengue = pd.read_csv(FEATURES_DENGUE_FILE)
print(f"features_flu: {features_flu.shape}")
print(f"features_dengue: {features_dengue.shape}")

📌 **[8.0]** Load feature CSVs đã tạo ở Session 7. Sau khi load, các biến `features_flu` và 
`features_dengue` sẵn sàng cho training — không cần chạy lại Session 7.

In [ ]:
# [8.1] Define feature columns (auto-detect tu column names)

# Exclude id cols va target cols
EXCLUDE_COLS = {'iso3', 'iso_year', 'iso_week', TARGET_FLU, TARGET_DENGUE,
                'rsv_cases', 'dengue_total', 'dengue_log1p', 'malaria_cases', 'malaria_log1p'}

FEATURE_COLS_FLU = [c for c in features_flu.columns
                    if c not in EXCLUDE_COLS and not c.startswith('dengue_lag')]

FEATURE_COLS_DENGUE = [c for c in features_dengue.columns
                       if c not in EXCLUDE_COLS and not c.startswith('inf_lag')]

print(f"FLU features ({len(FEATURE_COLS_FLU)}): {FEATURE_COLS_FLU[:5]}...")
print(f"DENGUE features ({len(FEATURE_COLS_DENGUE)}): {FEATURE_COLS_DENGUE[:5]}...")

📌 **[8.1]** Auto-detect feature columns thay vì hardcode để tránh lỗi nếu thêm/bớt feature ở Session 7. 
Logic: exclude các cột ID và target, và cross-exclude lag của bệnh kia 
(dengue_lag không vào flu model và ngược lại).

In [ ]:
# [8.2] Walk-forward CV function

def walk_forward_cv(df, feature_cols, target, val_years=range(2014, 2020)):
    """
    Walk-forward cross-validation cho time series.
    Moi fold: train tren tat ca nam truoc val_year, validate tren val_year.
    Returns: DataFrame voi fold, mae, rmse, mape, n_train, n_val.
    """
    results = []
    for val_year in val_years:
        train_mask = df['iso_year'] < val_year
        val_mask   = df['iso_year'] == val_year
        if val_mask.sum() == 0:
            continue
        X_train = df.loc[train_mask, feature_cols]
        y_train = df.loc[train_mask, target]
        X_val   = df.loc[val_mask, feature_cols]
        y_val   = df.loc[val_mask, target]
        model = XGBRegressor(
            n_estimators=300, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0
        )
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        preds = np.maximum(preds, 0)
        mae  = mean_absolute_error(y_val, preds)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        # sMAPE (symmetric — handles zero targets better than MAPE)
        smape = 100 * np.mean(2*np.abs(y_val - preds) / (np.abs(y_val) + np.abs(preds) + 1e-8))
        results.append({'fold': val_year, 'mae': round(mae,2), 'rmse': round(rmse,2),
                         'smape': round(smape,1),
                         'n_train': int(train_mask.sum()), 'n_val': int(val_mask.sum())})
    return pd.DataFrame(results)

print("walk_forward_cv function defined")

📌 **[8.2]** Walk-forward CV phù hợp time series hơn k-fold vì không có data leakage từ tương lai: 
mỗi fold chỉ train trên data trước val_year, validate trên val_year. 
6 folds (2014-2019) giúp đánh giá model ổn định trên nhiều năm khác nhau.

In [ ]:
# [8.3] Prophet baseline — Influenza (global aggregate, walk-forward CV)
from prophet import Prophet

def prophet_walk_forward(df, target, val_years=range(2014, 2020)):
    """Prophet baseline: global aggregate, walk-forward CV."""
    results = []
    global_df = df.groupby(['iso_year','iso_week'])[target].sum().reset_index()
    global_df['ds'] = pd.to_datetime(
        global_df['iso_year'].astype(str) + '-W' +
        global_df['iso_week'].astype(str).str.zfill(2) + '-1',
        format='%G-W%V-%u', errors='coerce'
    )
    global_df = global_df.dropna(subset=['ds']).rename(columns={target:'y'})
    for val_year in val_years:
        yr = global_df['ds'].dt.isocalendar().year
        train_df = global_df[yr < val_year][['ds','y']]
        val_df   = global_df[yr == val_year][['ds','y']]
        if len(val_df) == 0:
            continue
        m = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                    daily_seasonality=False, interval_width=0.95)
        m.fit(train_df)
        fc = m.predict(val_df[['ds']])
        y_true = val_df['y'].values
        y_pred = fc['yhat'].clip(0).values
        mae  = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        smape = 100 * np.mean(2*np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))
        results.append({'fold':val_year,'mae':mae,'rmse':rmse,'smape':smape,
                        'n_train':len(train_df),'n_val':len(val_df)})
    return pd.DataFrame(results)

print('=== PROPHET BASELINE — INFLUENZA (global aggregate) ===')
cv_prophet_flu = prophet_walk_forward(features_flu, TARGET_FLU)
print(cv_prophet_flu.to_string(index=False))
print(f'\nMean MAE:  {cv_prophet_flu["mae"].mean():,.0f}')
print(f'Mean RMSE: {cv_prophet_flu["rmse"].mean():,.0f}')
print(f'Mean MAPE: {cv_prophet_flu["mape"].mean():.1f}%')

📌 **[8.3]** Prophet dùng làm **baseline toàn cầu** (global aggregate) — không train per-country như XGBoost. Mục đích: đặt ngưỡng tối thiểu mà XGBoost cần vượt qua. Nếu XGBoost không thắng Prophet thì cần xem lại feature engineering. `clip(0)` vì Prophet có thể predict âm. Format `%G-W%V-%u` dùng ISO week-based year, tránh lệch 1 tuần ở đầu/cuối năm.

In [ ]:
# [8.4] Prophet baseline — Dengue
print('=== PROPHET BASELINE — DENGUE (global aggregate, log1p scale) ===')
cv_prophet_dengue = prophet_walk_forward(features_dengue, TARGET_DENGUE)
print(cv_prophet_dengue.to_string(index=False))
print(f'\nMean MAE:  {cv_prophet_dengue["mae"].mean():.4f}')
print(f'Mean RMSE: {cv_prophet_dengue["rmse"].mean():.4f}')
print(f'Mean sMAPE: {cv_prophet_dengue["smape"].mean():.1f}%')
print('(Note: target la dengue_log1p — MAE/RMSE o thang log)')

📌 **[8.4]** Dengue Prophet baseline train trên global aggregate của `dengue_log1p`. Global aggregate log1p không hoàn toàn có nghĩa toán học (sum log ≠ log sum) nhưng đủ để làm baseline so sánh. MAE/RMSE ở thang log — nhỏ hơn nhiều so với Influenza là bình thường.

In [ ]:
# [8.5] Walk-forward CV: Influenza
print("\n=== INFLUENZA WALK-FORWARD CV ===")
cv_flu = walk_forward_cv(features_flu, FEATURE_COLS_FLU, TARGET_FLU)
print(cv_flu.to_string(index=False))
print(f"\nMean MAE:  {cv_flu["mae"].mean():.2f}")
print(f"Mean RMSE: {cv_flu["rmse"].mean():.2f}")
print(f"Mean sMAPE: {cv_flu["smape"].mean():.1f}%")

📌 **[8.5]** Kết quả CV cho thấy model ổn định qua các năm. Fold có MAE cao nhất thường là năm 
có dịch bất thường (ví dụ pandemic year). sMAPE symmetric nên không bị explode khi denominator nhỏ — 
MAE và RMSE là metric chính; sMAPE dùng để so sánh cross-disease.

In [ ]:
# [8.6] Walk-forward CV: Dengue
print("\n=== DENGUE WALK-FORWARD CV ===")
cv_dengue = walk_forward_cv(features_dengue, FEATURE_COLS_DENGUE, TARGET_DENGUE)
print(cv_dengue.to_string(index=False))
print(f"\nMean MAE:  {cv_dengue["mae"].mean():.3f}")
print(f"Mean RMSE: {cv_dengue["rmse"].mean():.3f}")
print(f"Mean sMAPE: {cv_dengue["smape"].mean():.1f}%")
print("(Note: target la log1p-transformed dengue cases)")

📌 **[8.6]** Dengue target là `dengue_log1p` nên MAE/RMSE ở thang log. Để interpret: 
MAE=0.3 trên log1p tương đương ~e^0.3 = 1.35 lần sai số theo thang gốc. 
Model Dengue thường có variance cao hơn Influenza do tập trung ở ít quốc gia.

In [ ]:
# [8.7] Final model: train tren full 2010-2019

# Filter full train set
train_mask_flu    = (features_flu['iso_year'] >= TRAIN_START) & (features_flu['iso_year'] <= TRAIN_END)
train_mask_dengue = (features_dengue['iso_year'] >= TRAIN_START) & (features_dengue['iso_year'] <= TRAIN_END)

X_train_flu    = features_flu.loc[train_mask_flu, FEATURE_COLS_FLU]
y_train_flu    = features_flu.loc[train_mask_flu, TARGET_FLU]
X_train_dengue = features_dengue.loc[train_mask_dengue, FEATURE_COLS_DENGUE]
y_train_dengue = features_dengue.loc[train_mask_dengue, TARGET_DENGUE]

# Flu final model
if not MODEL_FLU_FILE.exists():
    xgb_flu = XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0
    )
    xgb_flu.fit(X_train_flu, y_train_flu)
    joblib.dump(xgb_flu, MODEL_FLU_FILE)
    print(f"✅ Saved xgb_flu_final.pkl")
else:
    xgb_flu = joblib.load(MODEL_FLU_FILE)
    print("⏭️  xgb_flu_final.pkl da ton tai, loaded")

# Dengue final model
if not MODEL_DENGUE_FILE.exists():
    xgb_dengue = XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0
    )
    xgb_dengue.fit(X_train_dengue, y_train_dengue)
    joblib.dump(xgb_dengue, MODEL_DENGUE_FILE)
    print(f"✅ Saved xgb_dengue_final.pkl")
else:
    xgb_dengue = joblib.load(MODEL_DENGUE_FILE)
    print("⏭️  xgb_dengue_final.pkl da ton tai, loaded")

📌 **[8.7]** Final model train trên full 2010-2019 (không chia val) để tận dụng toàn bộ data. 
Hyperparameters giữ nguyên theo CV để có so sánh fair. Idempotent: có file pkl rồi thì load lại.

In [ ]:
# [8.8] Feature importance: top 15 features moi model

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

for ax, model, feature_cols, title in [
    (axes[0], xgb_flu, FEATURE_COLS_FLU, 'Influenza — Top 15 Features'),
    (axes[1], xgb_dengue, FEATURE_COLS_DENGUE, 'Dengue — Top 15 Features'),
]:
    imp = pd.Series(model.feature_importances_, index=feature_cols)
    top15 = imp.nlargest(15).sort_values()
    top15.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Feature Importance (gain)')
    ax.set_ylabel('')

plt.tight_layout()
plt.show()

📌 **[8.8]** Feature importance giúp xác nhận: model có học đúng signal không? 
Kỳ vọng flu: `inf_lag1w`, `inf_lag2w` và `temp_c_flu_lag*` ở top. 
Kỳ vọng dengue: `dengue_lag8w`, `dengue_lag10w` và `precip_mm_dengue_lag*` ở top. 
Nếu feature nào không mong muốn nằm top, cần xem lại feature engineering.

In [ ]:
# [8.9] So sánh Prophet vs XGBoost — tổng kết

comparison = pd.DataFrame([
    {'Model':'Prophet (baseline)','Disease':'Influenza',
     'MAE':cv_prophet_flu['mae'].mean(),'RMSE':cv_prophet_flu['rmse'].mean(),
     'sMAPE(%)':cv_prophet_flu['smape'].mean(),'Scope':'Global aggregate'},
    {'Model':'XGBoost','Disease':'Influenza',
     'MAE':cv_flu['mae'].mean(),'RMSE':cv_flu['rmse'].mean(),
     'sMAPE(%)':cv_flu['smape'].mean(),'Scope':'Per-country features'},
    {'Model':'Prophet (baseline)','Disease':'Dengue',
     'MAE':cv_prophet_dengue['mae'].mean(),'RMSE':cv_prophet_dengue['rmse'].mean(),
     'sMAPE(%)':cv_prophet_dengue['smape'].mean(),'Scope':'Global aggregate'},
    {'Model':'XGBoost','Disease':'Dengue',
     'MAE':cv_dengue['mae'].mean(),'RMSE':cv_dengue['rmse'].mean(),
     'sMAPE(%)':cv_dengue['smape'].mean(),'Scope':'Per-country features'},
])
print('='*70)
print('SO SANH MODEL — Walk-forward CV mean (folds 2014–2019)')
print('='*70)
print(comparison.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#aec7e8', '#1f77b4']
for ax, metric in zip(axes, ['MAE','RMSE','sMAPE(%)']):
    for j, disease in enumerate(['Influenza','Dengue']):
        sub = comparison[comparison['Disease']==disease].reset_index(drop=True)
        ax.bar(np.arange(2) + j*0.02, sub[metric], 0.35,
               color=colors, alpha=0.85)
    ax.set_title(metric, fontsize=12)
    ax.set_xticks([0.175, 1.175])
    ax.set_xticklabels(['Prophet\n(baseline)', 'XGBoost'])
plt.suptitle('Prophet vs XGBoost — Walk-forward CV (2014–2019)', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('\n(SARIMA, LSTM → Future work)')

📌 **[8.9]** Bảng so sánh cuối SESSION 8. XGBoost dùng per-country lag + weather features nên kỳ vọng RMSE thấp hơn Prophet (chỉ dùng trend + seasonality toàn cầu). Nếu XGBoost thua Prophet cho một disease → cần xem lại feature engineering. **SARIMA** và **LSTM** được ghi nhận là future work: SARIMA phù hợp per-country nhưng chậm với 172 quốc gia; LSTM có thể capture non-linear temporal patterns tốt hơn nhưng cần nhiều GPU hơn.

# 📈 SESSION 9 — EVALUATION & EXPORT
> **Validate** trên 2022 (post-COVID generalization test)
> **Export** model artifacts và feature list cho FastAPI

In [ ]:
# [9.0] RESTART CELL — load models va features
xgb_flu    = joblib.load(MODEL_FLU_FILE)
xgb_dengue = joblib.load(MODEL_DENGUE_FILE)
features_flu    = pd.read_csv(FEATURES_FLU_FILE)
features_dengue = pd.read_csv(FEATURES_DENGUE_FILE)

EXCLUDE_COLS = {'iso3', 'iso_year', 'iso_week', TARGET_FLU, TARGET_DENGUE,
                'rsv_cases', 'dengue_total', 'dengue_log1p', 'malaria_cases', 'malaria_log1p'}
FEATURE_COLS_FLU = [c for c in features_flu.columns
                    if c not in EXCLUDE_COLS and not c.startswith('dengue_lag')]
FEATURE_COLS_DENGUE = [c for c in features_dengue.columns
                       if c not in EXCLUDE_COLS and not c.startswith('inf_lag')]

print("✅ Models va features loaded")
print(f"  FLU features: {len(FEATURE_COLS_FLU)}")
print(f"  DENGUE features: {len(FEATURE_COLS_DENGUE)}")

📌 **[9.0]** Restart cell: load model pkl và feature CSVs. Sau khi chạy cell này, 
toàn bộ Session 9 có thể chạy độc lập không cần Session 7 hay 8.

In [ ]:
# [9.1] Validate 2022: predict va tinh metrics

master_full = pd.read_csv(MASTER_FILE)

# Reconstruct feature dataframe cho 2022 (can co lag, nen dung features file nhung filter 2022)
# Strategy: rebuild features từ master (including 2022) cho validation
master_sorted = master_full.sort_values(['iso3', 'iso_year', 'iso_week']).reset_index(drop=True)
df_val = master_sorted.copy()

# Re-create lag features
for lag in LAG_FLU:
    df_val[f'inf_lag{lag}w'] = df_val.groupby('iso3')[TARGET_FLU].shift(lag)
for lag in LAG_DENGUE:
    df_val[f'dengue_lag{lag}w'] = df_val.groupby('iso3')[TARGET_DENGUE].shift(lag)
for window in [4, 8]:
    df_val[f'inf_roll{window}w'] = df_val.groupby('iso3')[TARGET_FLU].transform(
        lambda x: x.shift(1).rolling(window, min_periods=2).mean())
    df_val[f'dengue_roll{window}w'] = df_val.groupby('iso3')[TARGET_DENGUE].transform(
        lambda x: x.shift(1).rolling(window, min_periods=2).mean())
for var in ['temp_c', 'humidity_pct', 'precip_mm']:
    for lag in LAG_FLU:
        df_val[f'{var}_flu_lag{lag}w'] = df_val.groupby('iso3')[var].shift(lag)
for var in ['temp_c', 'precip_mm', 'humidity_pct']:
    for lag in [6, 8, 12]:
        df_val[f'{var}_dengue_lag{lag}w'] = df_val.groupby('iso3')[var].shift(lag)
df_val['sin_week'] = np.sin(2 * np.pi * df_val['iso_week'] / 52)
df_val['cos_week'] = np.cos(2 * np.pi * df_val['iso_week'] / 52)
df_val['quarter']  = ((df_val['iso_week'] - 1) // 13 + 1).clip(1, 4)

# Filter VAL_YEAR
val_flu    = df_val[df_val['iso_year'] == VAL_YEAR].dropna(subset=FEATURE_COLS_FLU + [TARGET_FLU])
val_dengue = df_val[(df_val['iso_year'] == VAL_YEAR) & (df_val['dengue_log1p'] > 0)].dropna(subset=FEATURE_COLS_DENGUE + [TARGET_DENGUE])

preds_flu    = np.maximum(xgb_flu.predict(val_flu[FEATURE_COLS_FLU]), 0)
preds_dengue = np.maximum(xgb_dengue.predict(val_dengue[FEATURE_COLS_DENGUE]), 0)

mae_flu  = mean_absolute_error(val_flu[TARGET_FLU], preds_flu)
rmse_flu = np.sqrt(mean_squared_error(val_flu[TARGET_FLU], preds_flu))
r2_flu   = r2_score(y_flu_true, y_flu_pred)
mae_den  = mean_absolute_error(val_dengue[TARGET_DENGUE], preds_dengue)
rmse_den = np.sqrt(mean_squared_error(val_dengue[TARGET_DENGUE], preds_dengue))

print(f"=== VALIDATION {VAL_YEAR} ===")
print(f"Influenza — MAE: {mae_flu:.2f} | RMSE: {rmse_flu:.2f} | n={len(val_flu)}")
print(f"Influenza — R² : {r2_flu:.2f} | RMSE: {rmse_flu:.2f} | n={len(val_flu)}")
print(f"Dengue    — MAE: {mae_den:.3f} | RMSE: {rmse_den:.3f} | n={len(val_dengue)}")

📌 **[9.1]** Validate 2022 là post-COVID generalization test — model được train trên 2010-2019 
(pre-COVID), không thấy 2020-2021 (COVID disruption), và predict 2022 (post-COVID return-to-normal). 
Nếu MAE 2022 gần với mean CV MAE (2014-2019) → model generalizes tốt qua COVID gap.
**R²** đo lường tỷ lệ variance được giải thích bởi model — R²>0.6 là chấp nhận được cho bài toán time-series dịch bệnh.

In [ ]:
# [9.2] Prediction vs Actual: global aggregate 2022

val_flu = val_flu.copy()
val_flu['pred'] = preds_flu
flu_agg = val_flu.groupby('iso_week')[[TARGET_FLU, 'pred']].sum()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
ax.plot(flu_agg.index, flu_agg[TARGET_FLU], label='Actual', color='steelblue', lw=2)
ax.plot(flu_agg.index, flu_agg['pred'], label='Predicted', color='tomato', lw=2, ls='--')
ax.set_title(f'Influenza 2022 — Global Aggregate (MAE={mae_flu:.0f})', fontsize=12)
ax.set_xlabel('ISO Week')
ax.set_ylabel('Total cases')
ax.legend()

val_dengue2 = val_dengue.copy()
val_dengue2['pred'] = preds_dengue
den_agg = val_dengue2.groupby('iso_week')[[TARGET_DENGUE, 'pred']].mean()

ax = axes[1]
ax.plot(den_agg.index, den_agg[TARGET_DENGUE], label='Actual', color='#27ae60', lw=2)
ax.plot(den_agg.index, den_agg['pred'], label='Predicted', color='#e67e22', lw=2, ls='--')
ax.set_title(f'Dengue 2022 — Mean log1p (MAE={mae_den:.3f})', fontsize=12)
ax.set_xlabel('ISO Week')
ax.set_ylabel('dengue_log1p')
ax.legend()

plt.tight_layout()
plt.show()

📌 **[9.2]** Biểu đồ Actual vs Predicted theo tuần. Nếu prediction bắt được seasonal peak 
(Influenza: tháng 12-2, Dengue: tháng 6-10 tùy vùng) → model có seasonality tốt. 
Lệch về amplitude (under/overpredict) thì điều chỉnh threshold risk classification.

In [ ]:
# [9.3] Risk classification: Low / Medium / High

def classify_risk(pred_series, train_preds):
    """Phan loai risk dua tren quantile cua training predictions."""
    low_q  = np.quantile(train_preds, 0.33)
    high_q = np.quantile(train_preds, 0.67)
    return pd.cut(
        pred_series,
        bins=[-np.inf, low_q, high_q, np.inf],
        labels=['Low', 'Medium', 'High']
    )

# Training predictions for quantile reference
train_flu_mask = (features_flu['iso_year'] >= TRAIN_START) & (features_flu['iso_year'] <= TRAIN_END)
train_preds_flu = xgb_flu.predict(features_flu.loc[train_flu_mask, FEATURE_COLS_FLU])

val_flu['risk'] = classify_risk(val_flu['pred'], train_preds_flu)
print("Influenza 2022 risk distribution:")
print(val_flu['risk'].value_counts())

# Sample: top 10 high-risk predictions
print("\nTop 10 high-risk (iso3, week, pred):")
print(val_flu[val_flu['risk']=='High'][['iso3','iso_week','pred',TARGET_FLU]]
      .sort_values('pred', ascending=False).head(10).to_string(index=False))
# Precision / Recall / F1 per risk tier
from sklearn.metrics import classification_report

def get_risk_tier(vals, low_thresh, high_thresh):
    return pd.cut(vals, bins=[-np.inf, low_thresh, high_thresh, np.inf],
                  labels=['Low','Medium','High'])

# Flu risk classification report
flu_low_t  = np.percentile(y_flu_true, 33)
flu_high_t = np.percentile(y_flu_true, 67)
risk_true_flu = get_risk_tier(y_flu_true, flu_low_t, flu_high_t)
risk_pred_flu = get_risk_tier(y_flu_pred, flu_low_t, flu_high_t)
print('=== FLU RISK CLASSIFICATION (2022 validation) ===')
print(classification_report(risk_true_flu, risk_pred_flu,
      target_names=['Low','Medium','High'], zero_division=0))

# Dengue risk classification report
dng_low_t  = np.percentile(y_dengue_true, 33)
dng_high_t = np.percentile(y_dengue_true, 67)
risk_true_dng = get_risk_tier(y_dengue_true, dng_low_t, dng_high_t)
risk_pred_dng = get_risk_tier(y_dengue_pred, dng_low_t, dng_high_t)
print('=== DENGUE RISK CLASSIFICATION (2022 validation) ===')
print(classification_report(risk_true_dng, risk_pred_dng,
      target_names=['Low','Medium','High'], zero_division=0))


📌 **[9.3]** Dùng quantile của training predictions thay vì absolute threshold vì scale ca bệnh 
khác nhau rất nhiều giữa các quốc gia (Brazil ~10,000 cases/tuần vs Laos ~50 cases/tuần). 
Quantile-based threshold đảm bảo ~33% quốc gia được classify High bất kể scale.
`classification_report` bên dưới cho thấy Precision/Recall/F1 per tier — đây là metric trực tiếp đánh giá khả năng cảnh báo đúng mức độ nguy cơ, quan trọng hơn RMSE đối với sản phẩm alert.

In [ ]:
# [9.4] Summary table

summary = pd.DataFrame([
    {
        'model': 'XGBoost Influenza',
        'target': TARGET_FLU,
        'val_year': VAL_YEAR,
        'val_MAE': round(mae_flu, 2),
        'val_RMSE': round(rmse_flu, 2),
        'n_features': len(FEATURE_COLS_FLU),
        'n_countries': val_flu['iso3'].nunique(),
        'train_period': f'{TRAIN_START}-{TRAIN_END}',
    },
    {
        'model': 'XGBoost Dengue',
        'target': TARGET_DENGUE,
        'val_year': VAL_YEAR,
        'val_MAE': round(mae_den, 3),
        'val_RMSE': round(rmse_den, 3),
        'n_features': len(FEATURE_COLS_DENGUE),
        'n_countries': val_dengue['iso3'].nunique(),
        'train_period': f'{TRAIN_START}-{TRAIN_END}',
    },
])

print('=== MODEL SUMMARY ===')
print(summary.T.to_string())
# R² và classification metrics — thêm vào summary
print('\n--- R² (2022 validation) ---')
print(f'Influenza R²: {r2_flu:.3f}')
print(f'Dengue    R²: {r2_dengue:.3f}')
print('(Precision/Recall/F1 — xem output [9.3] phia tren)')


📌 **[9.4]** Summary table là snapshot cuối cùng của toàn bộ ML pipeline — ghi lại vào Notion 
và dùng làm baseline comparison nếu sau này fine-tune hyperparameters.

In [ ]:
# [9.5] Export feature list to JSON (cho FastAPI)

feature_export = {
    'flu': {
        'target': TARGET_FLU,
        'features': FEATURE_COLS_FLU,
        'n_features': len(FEATURE_COLS_FLU),
        'lag_flu': LAG_FLU,
        'weather_vars': WEATHER_VARS,
    },
    'dengue': {
        'target': TARGET_DENGUE,
        'features': FEATURE_COLS_DENGUE,
        'n_features': len(FEATURE_COLS_DENGUE),
        'lag_dengue': LAG_DENGUE,
        'weather_vars': WEATHER_VARS,
    },
    'meta': {
        'train_period': f'{TRAIN_START}-{TRAIN_END}',
        'val_year': VAL_YEAR,
        'exclude_years': COVID_YEARS,
    }
}

with open(FEATURE_LIST_FILE, 'w') as f:
    json.dump(feature_export, f, indent=2)
print(f"✅ Saved feature list -> {FEATURE_LIST_FILE.name}")
print(f"  FLU: {len(FEATURE_COLS_FLU)} features")
print(f"  DENGUE: {len(FEATURE_COLS_DENGUE)} features")

📌 **[9.5]** Export feature list ra JSON để FastAPI backend biết cần nhận và xử lý những feature nào 
khi có request dự báo. File này là contract giữa ML pipeline và backend service — 
nếu thêm feature mới, regenerate file này và update API handler.